# ECE 491 — Python Lab 3: MNIST Digit Recognition Using a Perceptron
## Complete step-by-step Google Colab — Questions 1–14

**Original assignment:** “Python Lab 3_ECE 491.docx” (6 pages).

This notebook follows all **14 questions in the professor's sequence** and includes mathematical explanations, original MNIST loading, figures, working perceptron code, learning-rate experiments, validation/test accuracy, a final answer summary, and a theory cheat sheet.

### Important instructions from the handout

- Use the original MNIST dataset, **60,000 training images and 10,000 test images** (each 28 × 28 pixels).
- Keep only digit **0** and digit **1**.
- Convert **all image and label arrays to `float32`**, and vectorise label conversion to **−1/+1**.
- Train a **single-layer perceptron** with 784 pixel weights and report the validation and independent test accuracy.
- Investigate `eta`; stop if validation accuracy **exceeds 90%** at a 100-step checkpoint.
- Display and interpret the learned weight image.

### Professor's handout contains two inconsistencies

1. **Q6 (page 4):** The text asks to use the **last 20%** for validation, while the literal supplied code assigns the **first 80%** to validation and last 20% to training. This notebook defaults to **the professor's literal code**, and provides a documented switch to follow the textual 80/20 training/validation convention instead. Both are mathematically valid splits, but the sample counts are different.
2. **Q9 (pages 5–6):** The pseudocode prints accuracy in **percent** but sets a stopping threshold of **0.90**. This notebook uses **fractional accuracy** (0 to 1) internally and checks `accuracy > 0.90`, equivalent to more than 90% correct.

The professor does not specify fixed random weights or a single learning rate. Thus the **best tested eta, termination step and test accuracy are computed when you run the Colab**, not fabricated.

All displayed mathematics uses `$$ ... $$`.


## Theoretical foundation — how a perceptron recognises a digit

Each MNIST image has 28 rows and 28 columns:

$$
28\times28=784\text{ pixels}.
$$

Flattening the image gives a vector

$$
\mathbf{x}\in\mathbb{R}^{784}.
$$

The perceptron learns 784 weights:

$$
\mathbf{w}\in\mathbb{R}^{784}.
$$

It computes a weighted score:

$$
s=\mathbf{w}^{T}\mathbf{x}=\sum_{j=1}^{784}w_jx_j.
$$

For the lab's binary classes, use:

$$
\boxed{\hat y=\begin{cases}+1,&s\ge0\\-1,&s<0.\end{cases}}
$$

The label encoding is **digit 0 → −1** and **digit 1 → +1**.

**Analogy:** Each pixel casts a weighted vote. The sign of the combined votes decides whether the image looks more like a 0 or a 1.

# Setup — Obtain the four original MNIST `.gz` archives

The professor requires the original four compressed **MNIST IDX** files in a folder named `dataset/`:

- `train-images-idx3-ubyte.gz`
- `train-labels-idx1-ubyte.gz`
- `t10k-images-idx3-ubyte.gz`
- `t10k-labels-idx1-ubyte.gz`

The following cell offers **two input methods**:

- `DATA_SOURCE = "download"` (default): download the unmodified MNIST IDX files from a public Google-hosted MNIST mirror.
- `DATA_SOURCE = "upload"`: upload the four original `.gz` files from the professor manually in Google Colab.

Both methods use the same data-processing and testing code. No digit images or experimental results are invented.


In [ ]:
import gzip
import shutil
import struct
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 491
np.random.seed(SEED)

DATA_SOURCE = "download"  # Change to "upload" to upload the professor's four .gz files.
DATASET_DIR = Path("dataset")
DATASET_DIR.mkdir(exist_ok=True)

FILENAMES = [
    "train-images-idx3-ubyte.gz",
    "train-labels-idx1-ubyte.gz",
    "t10k-images-idx3-ubyte.gz",
    "t10k-labels-idx1-ubyte.gz",
]
BASE_URL = "https://storage.googleapis.com/cvdf-datasets/mnist/"

if DATA_SOURCE == "upload":
    from google.colab import files
    print("Upload all four original MNIST .gz archives in the same upload dialog.")
    uploaded = files.upload()
    missing = set(FILENAMES) - set(uploaded)
    if missing:
        raise ValueError(f"Missing original MNIST archives: {sorted(missing)}")
    for filename in FILENAMES:
        destination = DATASET_DIR / filename
        # files.upload() stores each file in the current Colab working directory.
        source = Path(filename)
        if source.resolve() != destination.resolve():
            shutil.copyfile(source, destination)
elif DATA_SOURCE == "download":
    for filename in FILENAMES:
        destination = DATASET_DIR / filename
        if not destination.exists() or destination.stat().st_size == 0:
            print("Downloading:", filename)
            try:
                urlretrieve(BASE_URL + filename, destination)
            except Exception as exc:
                if destination.exists():
                    destination.unlink()
                raise RuntimeError(
                    f"Cannot download {filename}. Switch DATA_SOURCE='upload' "
                    "and upload all four original MNIST .gz files."
                ) from exc
else:
    raise ValueError("DATA_SOURCE must be 'download' or 'upload'.")

for filename in FILENAMES:
    destination = DATASET_DIR / filename
    print(f"{filename}: {destination.stat().st_size:,} bytes")

NROWS = NCOLS = 28
MNIST_TRAIN_IMS_GZ = DATASET_DIR / FILENAMES[0]
MNIST_TRAIN_LBS_GZ = DATASET_DIR / FILENAMES[1]
MNIST_TEST_IMS_GZ = DATASET_DIR / FILENAMES[2]
MNIST_TEST_LBS_GZ = DATASET_DIR / FILENAMES[3]


## Loading the binary MNIST files

The original lab gives a `load_data()` function. This equivalent implementation reads the IDX headers, checks the file type and shape, and returns the same four arrays. Reading with `np.frombuffer` avoids constructing an enormous Python tuple of pixel values.

In [ ]:
def read_mnist_images(path):
    with gzip.open(path, "rb") as f:
        header = f.read(16)
        if len(header) != 16:
            raise ValueError(f"Invalid image header: {path}")
        magic, count, rows, cols = struct.unpack(">IIII", header)
        if magic != 2051 or (rows, cols) != (28, 28):
            raise ValueError(f"Unexpected MNIST image header in {path}")
        raw = f.read()
    pixels = np.frombuffer(raw, dtype=np.uint8)
    if pixels.size != count * rows * cols:
        raise ValueError(f"Incorrect pixel count in {path}")
    return pixels.reshape(count, rows * cols).copy()

def read_mnist_labels(path):
    with gzip.open(path, "rb") as f:
        header = f.read(8)
        if len(header) != 8:
            raise ValueError(f"Invalid label header: {path}")
        magic, count = struct.unpack(">II", header)
        if magic != 2049:
            raise ValueError(f"Unexpected MNIST label header in {path}")
        raw = f.read()
    labels = np.frombuffer(raw, dtype=np.uint8)
    if labels.size != count:
        raise ValueError(f"Incorrect label count in {path}")
    return labels.copy()

def load_data():
    return (
        read_mnist_images(MNIST_TRAIN_IMS_GZ),
        read_mnist_labels(MNIST_TRAIN_LBS_GZ),
        read_mnist_images(MNIST_TEST_IMS_GZ),
        read_mnist_labels(MNIST_TEST_LBS_GZ),
    )

train_ims, train_lbs, test_ims, test_lbs = load_data()
print("Original MNIST loaded successfully.")

# Question 1 — Shapes of all **four** MNIST arrays

The professor explicitly asks to print **all four array shapes**. Flattening a 28 × 28 image produces 784 pixels.

Before filtering, the standard shapes are:

$$
\boxed{\mathrm{train\_ims.shape}=(60000,784)}
$$

$$
\boxed{\mathrm{train\_lbs.shape}=(60000,)}
$$

$$
\boxed{\mathrm{test\_ims.shape}=(10000,784)}
$$

$$
\boxed{\mathrm{test\_lbs.shape}=(10000,)}
$$


In [ ]:
print("Q1 — Four original array shapes:")
print("train_ims:", train_ims.shape)
print("train_lbs:", train_lbs.shape)
print("test_ims: ", test_ims.shape)
print("test_lbs: ", test_lbs.shape)

assert train_ims.shape == (60000, 784)
assert train_lbs.shape == (60000,)
assert test_ims.shape == (10000, 784)
assert test_lbs.shape == (10000,)


# Question 2 — Display the fifth training image and its label

Python indexing begins at zero, so the fifth example is **index 4**.

Reshape its 784-element vector back into a 28-by-28 image. The label printed underneath should match the digit visible in the picture.

In [ ]:
index = 4
plt.figure(figsize=(3.5, 3.5))
plt.imshow(train_ims[index].reshape(28, 28), cmap="gray")
plt.title(f"Fifth training image — label: {train_lbs[index]}")
plt.axis("off")
plt.show()
print("Q2 — Fifth training label:", int(train_lbs[index]))

# Question 3 — Repeat Q1 and Q2 for the test set

The independent MNIST test set contains 10,000 images.

Expected shapes:

$$
\boxed{\mathrm{test\_ims.shape}=(10000,784)}
$$

$$
\boxed{\mathrm{test\_lbs.shape}=(10000,)}
$$

In [ ]:
print("Q3 — Test image shape:", test_ims.shape)
print("Q3 — Test label shape:", test_lbs.shape)
assert test_ims.shape == (10000, 784)
assert test_lbs.shape == (10000,)

index = 4
plt.figure(figsize=(3.5, 3.5))
plt.imshow(test_ims[index].reshape(28, 28), cmap="gray")
plt.title(f"Fifth test image — label: {test_lbs[index]}")
plt.axis("off")
plt.show()
print("Q3 — Fifth test label:", int(test_lbs[index]))

# Question 4 — Keep only training digits 0 and 1

The lab asks us to use logical indexing:

$$
\mathrm{mask}=(y=0)\lor(y=1).
$$

The original MNIST training set has 5,923 zeros and 6,742 ones:

$$
\boxed{5923+6742=12665\text{ training examples}}.
$$

In [ ]:
train_mask = np.logical_or(train_lbs == 0, train_lbs == 1)
train_ims = train_ims[train_mask, :]
train_lbs = train_lbs[train_mask]
print("Q4 — Training examples after filtering:", len(train_lbs))
print("Number of 0s:", np.count_nonzero(train_lbs == 0))
print("Number of 1s:", np.count_nonzero(train_lbs == 1))
assert len(train_lbs) == 12665

# Question 5 — Keep only test digits 0 and 1

The original test set has 980 zeros and 1,135 ones:

$$
\boxed{980+1135=2115\text{ test examples}}.
$$

In [ ]:
test_mask = np.logical_or(test_lbs == 0, test_lbs == 1)
test_ims = test_ims[test_mask, :]
test_lbs = test_lbs[test_mask]
print("Q5 — Test examples after filtering:", len(test_lbs))
print("Number of 0s:", np.count_nonzero(test_lbs == 0))
print("Number of 1s:", np.count_nonzero(test_lbs == 1))
assert len(test_lbs) == 2115

# Question 6 — Training and validation sample counts

The original filtered training set contains **12,665** examples. The handout is internally inconsistent:

- **Professor's literal code (default):** `val_ims = train_ims[:int(0.8*N)]`, then `train_ims = train_ims[int(0.8*N):]`. This produces **10,132 validation** examples and **2,533 training** examples.
- **Written instruction:** "taking the last 20% of the training dataset" for validation. This produces **10,132 training** and **2,533 validation** examples.

To preserve the exact provided code, the default is `SPLIT_RULE = "professor_code"`. The alternative `"written_instruction"` follows the handout's English description.

Do **not** mix these two conventions or report one count while running the other.


In [ ]:
# Choose the rule used in the professor's actual example code by default.
SPLIT_RULE = "professor_code"  # Alternative: "written_instruction"

split = int(0.8 * len(train_lbs))

if SPLIT_RULE == "professor_code":
    # Literal code in the professor's Lab 3 document (page 4).
    val_ims = train_ims[:split, :].copy()
    val_lbs = train_lbs[:split].copy()
    train_ims = train_ims[split:, :].copy()
    train_lbs = train_lbs[split:].copy()
elif SPLIT_RULE == "written_instruction":
    # Alternative: last 20% used as validation, as described in the text.
    val_ims = train_ims[split:, :].copy()
    val_lbs = train_lbs[split:].copy()
    train_ims = train_ims[:split, :].copy()
    train_lbs = train_lbs[:split].copy()
else:
    raise ValueError("SPLIT_RULE must be 'professor_code' or 'written_instruction'.")

num_training_samples = train_ims.shape[0]
num_val_samples = val_ims.shape[0]
num_test_samples = test_ims.shape[0]

print("Q6 — Split rule:", SPLIT_RULE)
print("Q6 — Training examples:", num_training_samples)
print("Q6 — Validation examples:", num_val_samples)
print("Independent test examples:", num_test_samples)

if SPLIT_RULE == "professor_code":
    assert (num_training_samples, num_val_samples, num_test_samples) == (2533, 10132, 2115)
else:
    assert (num_training_samples, num_val_samples, num_test_samples) == (10132, 2533, 2115)


# Question 7 — Convert all arrays to `float32`

`uint8` is an unsigned integer type. The professor requests floating-point arrays and label vectors:

$$
\boxed{\mathrm{dtype}=\texttt{float32}}.
$$

**Important:** Do not change the pixel scale here; the handout asks for type casting, not image normalization.

In [ ]:
train_ims = train_ims.astype(np.float32)
train_lbs = train_lbs.astype(np.float32)
val_ims = val_ims.astype(np.float32)
val_lbs = val_lbs.astype(np.float32)
test_ims = test_ims.astype(np.float32)
test_lbs = test_lbs.astype(np.float32)

for name, arr in [
    ("train_ims", train_ims), ("train_lbs", train_lbs),
    ("val_ims", val_ims), ("val_lbs", val_lbs),
    ("test_ims", test_ims), ("test_lbs", test_lbs)
]:
    print(f"Q7 — {name}: {arr.dtype}")
    assert arr.dtype == np.float32

# Question 8 — Convert labels from {0, 1} to {−1, +1}

The perceptron uses signed labels:

$$
\boxed{y_{\mathrm{signed}}=2y_{\mathrm{original}}-1}.
$$

Check:

$$
0\mapsto-1,\qquad 1\mapsto+1.
$$

This is a vectorised operation; **no `for` loop** is needed.

In [ ]:
train_lbs = 2.0 * train_lbs - 1.0
val_lbs = 2.0 * val_lbs - 1.0
test_lbs = 2.0 * test_lbs - 1.0

for name, labels in [
    ("train_lbs", train_lbs),
    ("val_lbs", val_lbs),
    ("test_lbs", test_lbs)
]:
    print(f"Q8 — {name}: {np.unique(labels)}")
    assert set(np.unique(labels)) == {-1.0, 1.0}

# Question 9 — Complete the perceptron algorithm

The professor's prediction formula is

$$
\hat y=\mathrm{sign}(\mathbf w^T\mathbf x).
$$

To avoid the ambiguous value 0 returned by `np.sign(0)`, assign zero scores to the +1 class.

For each training example:

$$
e=y_{\mathrm{true}}-\hat y
$$

and update:

$$
\boxed{\mathbf w\leftarrow\mathbf w+\eta e\mathbf x}.
$$

The lab asks for validation every 100 steps and early stopping when validation accuracy exceeds 90%.

The function below implements **exactly this error-based update** and checks validation accuracy at indices 0, 100, 200, and so on.

The random initial weights are sampled once and reused in each learning-rate experiment, so comparisons are reproducible.

In [ ]:
num_training_samples = train_ims.shape[0]
num_val_samples = val_ims.shape[0]
num_test_samples = test_ims.shape[0]

def predict_perceptron(X, weights):
    scores = np.asarray(X) @ weights
    return np.where(scores >= 0, 1.0, -1.0).astype(np.float32)

def accuracy_fraction(X, labels, weights):
    return float(np.mean(predict_perceptron(X, weights) == labels))

def train_perceptron(eta, initial_weights, verbose=True):
    if eta <= 0:
        raise ValueError("eta must be positive")
    weights = initial_weights.copy()
    history = []
    stop_index = None

    for idx in range(num_training_samples):
        x = train_ims[idx, :]
        y_true = train_lbs[idx]
        y_pred = 1.0 if float(np.dot(weights, x)) >= 0 else -1.0
        error = y_true - y_pred
        update = eta * error * x
        weights += update

        if idx % 100 == 0:
            accuracy = accuracy_fraction(val_ims, val_lbs, weights)
            history.append((idx, accuracy))
            if verbose:
                print(f"step: {idx:5d}, validation accuracy: {100*accuracy:.2f}%")
            if accuracy > 0.90:
                stop_index = idx
                break

    return {
        "eta": eta,
        "weights": weights,
        "history": history,
        "stop_index": stop_index,
        "validation_accuracy": accuracy_fraction(val_ims, val_lbs, weights),
    }

initial_weights = np.random.default_rng(SEED).normal(
    0.0, 1.0, size=NROWS * NCOLS
).astype(np.float32)

print("Q9 — Training samples:", num_training_samples)
print("Q9 — Validation samples:", num_val_samples)
print("Q9 — Weight vector shape:", initial_weights.shape)

# Question 10 — Experiment with the learning rate

The lab does **not** prescribe one universally correct learning rate.

Try several positive values, using the same initial weights and validation data. Select the learning rate with the highest measured validation accuracy; use the smallest tested learning rate to break a tie.

Because the pixels are not normalised (values from 0 to 255), the numerical scale of the updates depends on `eta`. Report the **actual experiment** rather than claiming a value without testing it.

In [ ]:
eta_candidates = [1e-7, 1e-6, 1e-5, 1e-4, 1e-3]
experiments = []

for eta in eta_candidates:
    result = train_perceptron(eta, initial_weights, verbose=False)
    experiments.append(result)
    print(
        f"eta={eta:.0e} | "
        f"validation accuracy={100*result['validation_accuracy']:.2f}% | "
        f"stop index={result['stop_index']}"
    )

best = max(
    experiments,
    key=lambda result: (result["validation_accuracy"], -result["eta"])
)
weights = best["weights"]
best_eta = best["eta"]

print("\nQ10 — Selected eta:", best_eta)
print("Validation accuracy:", f"{100*best['validation_accuracy']:.2f}%")

display(pd.DataFrame([
    {
        "eta": r["eta"],
        "validation_accuracy_percent": 100 * r["validation_accuracy"],
        "stop_index": r["stop_index"]
    }
    for r in experiments
]))

# Question 11 — At which step did the algorithm terminate?

The pseudocode checks validation accuracy at

$$
0,100,200,\ldots
$$

and stops at the first checked step with

$$
\boxed{\mathrm{accuracy}>0.90}.
$$

Since the professor uses a random initial vector and lets us choose the learning rate, the stopping step is **experimental**, not a fixed theoretical number. The following cell reports it accurately.

In [ ]:
if best["stop_index"] is not None:
    print("Q11 — Stopped at zero-based step index:", best["stop_index"])
    print("Training examples processed:", best["stop_index"] + 1)
else:
    print("Q11 — Did not exceed 90% at a validation checkpoint.")
    print("Completed training examples:", num_training_samples)

# Question 12 — Accuracy on the independent test set

The test data were **not** used to train the perceptron or select `eta`.

Test accuracy is:

$$
\boxed{
\mathrm{Accuracy}
=
\frac{\text{number of correctly classified test images}}
{\text{total number of test images}}\times100\%.
}
$$

There are 2,115 filtered test examples. The exact accuracy depends on the trained weights and is calculated below.

In [ ]:
test_predictions = predict_perceptron(test_ims, weights)
correct = int(np.count_nonzero(test_predictions == test_lbs))
test_accuracy = 100.0 * correct / num_test_samples

print("Q12 — Correct test predictions:", correct)
print("Q12 — Total test images:", num_test_samples)
print(f"Q12 — Test accuracy: {test_accuracy:.2f}%")

assert 0 <= test_accuracy <= 100

# Question 13 — Visualise the learned weight vector

The learned vector has 784 elements. Reshape it into a 28-by-28 image:

$$
\boxed{\mathbf w\in\mathbb R^{784}
\longrightarrow
W\in\mathbb R^{28\times28}.}
$$

To reveal both positive and negative weights, use a symmetric colour scale centred on zero.

- **Positive weights:** pixels that push the decision towards digit **1**.
- **Negative weights:** pixels that push the decision towards digit **0**.

In [ ]:
weight_image = weights.reshape(NROWS, NCOLS)
limit = float(np.max(np.abs(weight_image)))
if limit == 0:
    limit = 1.0

plt.figure(figsize=(6, 5))
plt.imshow(weight_image, cmap="bwr", vmin=-limit, vmax=limit)
plt.colorbar(label="Learned perceptron weight")
plt.title("Q13 — Learned 28×28 Perceptron Weight Map")
plt.axis("off")
plt.show()

# Question 14 — Interpret the weight visualisation

**Suggested professor-style explanation:**

The visualised 28-by-28 matrix shows which image pixels contribute most strongly to the perceptron's decision. Positive weights favour class **+1 (digit 1)**, while negative weights favour class **−1 (digit 0)**. Because digit 1 often contains a narrow vertical stroke and digit 0 often contains a rounded loop, the learned weight map may show a spatial pattern that helps distinguish these shapes.

The weight map is **not itself a handwritten digit**. It is a linear decision template learned from many examples. The exact colours and patterns depend on the training data, learning rate, and random initial weights.

**Conclusion:** The perceptron classifies each image by taking the sign of the weighted sum of its pixels. Its success is measured using the independent test accuracy from Question 12.

# Final answer sheet — Generated from your actual run

This cell prints the quantities the professor asks you to report, without inventing learning-rate, stopping-step, or test-accuracy results.

In [ ]:
print("ECE 491 — Python Lab 3 | Final Results")
print("-" * 55)
print("Q1: Training image shape:", (60000, 784))
print("Q1: Training label shape:", (60000,))
print("Q2: Fifth training label:", int(read_mnist_labels(MNIST_TRAIN_LBS_GZ)[4]))
print("Q3: Test image shape:", (10000, 784))
print("Q3: Test label shape:", (10000,))
print("Q3: Fifth test label:", int(read_mnist_labels(MNIST_TEST_LBS_GZ)[4]))
print("Q4: Filtered training examples:", 12665)
print("Q5: Filtered test examples:", 2115)
print("Q6: Training / validation:", num_training_samples, num_val_samples)
print("Q7: Image and label arrays are float32")
print("Q8: Labels are -1 and +1")
print("Q9: Perceptron implemented and trained")
print("Q10: Best tested eta:", best_eta)
print("Q11: Stop index:", best["stop_index"])
print(f"Q12: Test accuracy: {test_accuracy:.2f}%")
print("Q13: Learned weight image plotted above")
print("Q14: Interpretation provided above")

# Final theory cheat sheet — Lab 3

| Concept | What to remember |
|---|---|
| MNIST | 70,000 handwritten digit images: 60,000 train, 10,000 test |
| Image size | 28 × 28 = 784 pixels |
| Flattening | A 2D image becomes a 784-dimensional vector |
| Binary classification | Keep only digits 0 and 1 |
| Logical indexing | Select samples with `(labels == 0) | (labels == 1)` |
| Validation set | Used for learning-rate selection and stopping |
| Test set | Used only for the final independent evaluation |
| `float32` | Floating-point numerical representation |
| Label mapping | Digit 0 → −1; digit 1 → +1 |
| Weight vector | 784 learned coefficients |
| Prediction | Sign of the weighted pixel sum |
| Error | True signed label minus predicted signed label |
| Learning rate | Controls the size of each update |
| Accuracy | Correct predictions divided by total predictions |
| Weight visualisation | Reveals pixels favouring each class |

## The four essential formulas

**1. Linear score**

$$
s=\mathbf w^T\mathbf x.
$$

**2. Prediction**

$$
\hat y=\begin{cases}+1,&s\ge0\\-1,&s<0.\end{cases}
$$

**3. Weight update (the lab's specified form)**

$$
\boxed{\mathbf w\leftarrow\mathbf w+\eta(y-\hat y)\mathbf x.}
$$

**4. Accuracy**

$$
\boxed{\mathrm{Accuracy}=\frac{\mathrm{Correct}}{\mathrm{Total}}\times100\%.}
$$

## Interview-style understanding checks

**Why is the test set kept separate?** Because evaluating on images used to tune the model would give an overly optimistic estimate of how well the perceptron handles unseen images.

**What does the weight image represent?** It represents a learned linear pattern of pixel importance for distinguishing digit 0 from digit 1.

**Why can the stopping step change?** Different initial weights and learning rates change the path of the training algorithm, and therefore when validation accuracy first exceeds the threshold.

## Source-alignment checklist

- Q1 prints **all four** shapes; Q3 repeats the check and displays the fifth test image.
- Q2 uses Python index `4` for the fifth training image; `label_ims` is a typo in the handout for `train_lbs`.
- Q4–Q5 filter the real MNIST set to digits 0 and 1; Q6 discloses both conflicting split conventions and uses the actual professor code by default.
- Q7 casts **all six** resulting arrays (training, validation and test, images and labels) to `float32`.
- Q8 maps 0 to −1 and 1 to +1 **without a loop**.
- Q9–Q12 use the professor's **784-weight perceptron**, validation checks, learning-rate experiments, termination rule and independent test accuracy.
- Q13–Q14 visualise and interpret the signed weight map.
